# Proyecto Parte 2: Clasificación de sentimiento multilingüe con Deep Learning y aumentación de datos

## Miembros del grupo 45

- Juan Diego Ramírez Garzón - 202410855
- Sebastián Vargas Otero - 202323732
- Mateo Zambrano Burbano - 202321531
- Nicolás Mario Romero Colmenares - 202321309

### Resumen del enfoque

| Etapa | Decisión |
|---|---|
| Framework | TensorFlow + Keras 3 + **KerasHub** (modelos preentrenados en Keras) |
| Modelo | **XLM-RoBERTa** (*encoder-only*, preentrenado en 100 idiomas): tamaño *base* para comparar configuraciones y tamaño ***large*** para el modelo final |
| Modelo final | **Ensamble** (promedio de probabilidades) de XLM-R large entrenados con semillas distintas, si supera al mejor modelo individual en validación |
| Línea de comparación | BiLSTM entrenada desde cero (sin preentrenamiento) |
| Aumentación 1 | **Code-switching léxico**: sustitución de palabras es→en y en→es con los diccionarios bilingües MUSE (no se usa ningún modelo generativo) |
| Aumentación 2 | **Inserción de oraciones neutras de otro idioma**: se intercalan oraciones descriptivas (sin opinión) tomadas de reseñas neutrales en otro idioma |
| Restricción | Por el anuncio del 12/11/2026 solo se usan modelos *encoder-only* (XLM-R) con cabeza de clasificación, y ningún modelo generativo, ni siquiera para aumentar datos |
| Validación | Partición *hold-out* 90/10 estratificada por **idioma × clase**; la aumentación solo se aplica a la partición de entrenamiento |
| Métrica | Exactitud (*accuracy*), igual que Kaggle, global y **por idioma** |

### Cómo ejecutarlo (Kaggle)

1. *Add Input* → la competencia del curso (aparece en `/kaggle/input/...`).
2. *Settings* → **Accelerator: GPU T4 x2** y **Internet: On** (para descargar los modelos preentrenados y los diccionarios).
3. **Save Version → Save & Run All (Commit)**. Así corre completo en segundo plano (hasta 12 h) aunque se cierre el navegador, y los archivos de `envios/` y `modelos/` quedan en la pestaña *Output* de esa versión.
4. Para probar interactivamente, *Run All* también funciona: las predicciones de cada experimento se guardan en `/kaggle/working` y, si la sesión sigue viva, se puede reanudar sin repetir lo ya calculado (ver `REUTILIZAR_CACHE`).

Tiempo aproximado con GPU T4: aumentación < 1 min, BiLSTM ≈ 5 min, cada XLM-R base ≈ 20-35 min, cada XLM-R large ≈ 60-90 min. Total ≈ 4-5 horas.

### Contenido

0. Configuración del entorno
1. Carga y calidad de datos
2. Exploración: idiomas, clases, longitud y tipos de reseña difíciles
3. Protocolo de validación
4. Aumentación de datos (code-switching léxico e inserción de oraciones neutras)
5. Experimentos (BiLSTM y XLM-RoBERTa, con y sin aumentación)
6. Comparación con y sin aumentación
7. Análisis de errores por idioma
8. Modelo final, guardado y envío
9. Conclusiones

### **0. Configuración del entorno**

In [ ]:
import os, re, gc, json, math, time, random, warnings, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)

CLASES = ['negativo', 'neutral', 'positivo']
C2I = {c: i for i, c in enumerate(CLASES)}

# Rutas: funciona igual en Kaggle y en local
EN_KAGGLE = Path('/kaggle/input').exists()
if EN_KAGGLE:
    DATA = next(p.parent for p in Path('/kaggle/input').rglob('train.csv') if (p.parent / 'eval.csv').exists())
    TRABAJO = Path('/kaggle/working')
else:
    DATA = Path('Data')
    TRABAJO = Path('.')

DIR_AUG = TRABAJO / 'aumentacion'
DIR_MOD = TRABAJO / 'modelos'
DIR_ENV = TRABAJO / 'envios'
DIR_RES = TRABAJO / 'resultados'
for d in (DIR_AUG, DIR_MOD, DIR_ENV, DIR_RES):
    d.mkdir(parents=True, exist_ok=True)

# Si ya existen predicciones guardadas de experimentos anteriores, se reutilizan en lugar de recalcularlas
REUTILIZAR_CACHE = True
print('Datos en:', DATA.resolve())

In [ ]:
import tensorflow as tf

GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)   # TF reserva memoria de GPU a medida que la necesita

import keras
try:
    import keras_hub
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub'], check=True)
    import keras_hub

keras.utils.set_random_seed(SEMILLA)
# Precisión mixta: casi duplica la velocidad en GPUs con tensor cores (T4) y reduce memoria
keras.mixed_precision.set_global_policy('mixed_float16' if GPUS else 'float32')

print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| KerasHub', keras_hub.__version__)
print('GPUs:', GPUS)

### **1. Carga y calidad de datos**

Se repite la auditoría de la Parte 1 (nulos, duplicados, etiquetas en conflicto y fuga entre `train` y `eval`), porque el conjunto de entrenamiento creció de 12,000 a 20,466 reseñas y ahora mezcla idiomas.

In [ ]:
train_raw = pd.read_csv(DATA / 'train.csv')
eval_raw = pd.read_csv(DATA / 'eval.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
print('train:', train_raw.shape, '| eval:', eval_raw.shape)
display(train_raw.head())

In [ ]:
def clave(s):
    return s.fillna('').str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

train = train_raw.copy()
train['label'] = train['label'].astype(str).str.strip().str.lower()
train['clave'] = clave(train['text'])
eval_df = eval_raw.copy()
eval_df['text'] = eval_df['text'].fillna('')

grupos = train.groupby('clave')['label'].agg(['size', 'nunique'])
print('Nulos train:', train_raw.isna().sum().to_dict(), '| nulos eval:', eval_raw.isna().sum().to_dict())
print('Etiquetas fuera de las 3 clases:', sorted(set(train['label']) - set(CLASES)) or 'ninguna')
print('Duplicados (clave laxa):', int(train['clave'].duplicated().sum()))
print('Textos repetidos con etiquetas distintas:', int((grupos['nunique'] > 1).sum()))
print('Reseñas de eval que aparecen en train:', int(clave(eval_df['text']).isin(train['clave']).sum()))

# Limpieza: se quitan conflictos y duplicados para evitar fuga entre train y validación
n0 = len(train)
train = train[train['clave'] != '']
train = train[~train['clave'].isin(grupos[grupos['nunique'] > 1].index)]
train = train.drop_duplicates('clave').drop(columns='clave').reset_index(drop=True)
train['y'] = train['label'].map(C2I)
print(f'train: {n0:,} -> {len(train):,} filas')

### **2. Exploración**

#### 2.1 Detección de idioma

El idioma no viene en los datos, pero lo necesitamos para (i) estratificar la validación, (ii) decidir qué reseñas aumentar y (iii) analizar los errores por idioma, como pide el enunciado.

Se usa un detector propio basado en **palabras funcionales** (artículos, preposiciones, conectores) propias de cada idioma. Se eligió así porque no requiere descargar nada, es transparente y para cuatro idiomas cercanos funciona bien. Una reseña se marca como **`es-en` (code-switching)** cuando tiene al menos 3 palabras funcionales de español y 3 de inglés, y el idioma minoritario aporta al menos el 20 %.

*Limitación:* es una heurística. Se valida manualmente con ejemplos más abajo.

In [ ]:
PALABRAS_IDIOMA = {
    'es': 'el los las que y es por para con muy pero como más mi del ya está fue eso todo cuando lo se me una al le hasta desde porque también llegó compré',
    'en': 'the and is it to of for that was with my but not this have be so you its just very really i bought arrived works great',
    'pt': 'os em um é não do da na mas muito meu minha pra eu tem foi isso chegou comprei também uma até depois veio',
    'fr': "le les et est une pour pas du des je mais très il mon ce qui avec sur ça j'ai acheté arrivé",
}
PALABRAS_IDIOMA = {k: set(v.split()) for k, v in PALABRAS_IDIOMA.items()}
TOKEN_RE = re.compile(r"[a-záéíóúàâçêôãõüñèëîïû']+")

def detectar_idioma(texto):
    palabras = TOKEN_RE.findall(texto.lower())
    c = {k: sum(p in s for p in palabras) for k, s in PALABRAS_IDIOMA.items()}
    es, en = c['es'], c['en']
    if es >= 3 and en >= 3 and min(es, en) / (es + en) >= 0.2:
        return 'es-en'
    return max(c, key=c.get)

IDIOMAS = ['es', 'en', 'pt', 'fr', 'es-en']
train['idioma'] = train['text'].map(detectar_idioma)
eval_df['idioma'] = eval_df['text'].map(detectar_idioma)

dist = pd.concat([train['idioma'].value_counts(normalize=True).rename('train'),
                  eval_df['idioma'].value_counts(normalize=True).rename('eval')], axis=1).reindex(IDIOMAS).fillna(0)
display((dist * 100).round(1).style.format('{:.1f} %').set_caption('Distribución de idiomas'))

for idi in IDIOMAS:
    print(f'== {idi}')
    for t in train.loc[train['idioma'] == idi, 'text'].sample(min(2, (train['idioma'] == idi).sum()), random_state=1):
        print('  -', t[:160], '...')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
(dist * 100).plot.bar(ax=axes[0], rot=0, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Idiomas: train vs eval (%)')

pd.crosstab(train['idioma'], train['label'], normalize='index').reindex(IDIOMAS).plot.bar(
    stacked=True, ax=axes[1], rot=0, color=['#E74C3C', '#95A5A6', '#2ECC71'])
axes[1].set_title('Clases por idioma (train)')
axes[1].legend(loc='lower right', fontsize=8)

train['n_palabras'] = train['text'].str.split().str.len()
eval_df['n_palabras'] = eval_df['text'].str.split().str.len()
sns.kdeplot(train['n_palabras'], ax=axes[2], label='train')
sns.kdeplot(eval_df['n_palabras'], ax=axes[2], label='eval')
axes[2].set_title('Longitud (palabras)')
axes[2].legend()
plt.tight_layout(); plt.show()

display(train['label'].value_counts().reindex(CLASES).to_frame('n'))

#### 2.2 Tipos de reseña difíciles

El enunciado advierte de reseñas con ironía, negaciones de largo alcance, comparaciones, veredicto al inicio o que solo describen consecuencias. Se marcan con expresiones regulares multilingües (aproximadas) para comparar su frecuencia en `train` y `eval`, y luego para medir el error en cada tipo (sección 7).

In [ ]:
PATRONES = {
    'negacion_larga': r"\b(no (diría|diria|creo que|es que|puedo decir)|tampoco|i wouldn'?t say|i can'?t say|not that it|não diria|nao diria|não é que|je ne dirais|ce n'est pas que)",
    'comparacion':    r"\b(esperaba|expected|esperava|attendais|mejor que|peor que|better than|worse than|melhor que|pior que|mieux que|moins bien que)",
    'consecuencia':   r"(devolv|reembols|returned it|refund|sent it back|send it back|devolvi|rendu|renvoy|rembours)",
    'ironia':         r"(qué maravilla|que maravilla|claro que sí|just great|oh great|yeah right|que maravilha|quelle merveille|bravo|genial, |perfecto, |great, )",
    'contraste':      r"\b(pero|aunque|sin embargo|but|however|although|porém|contudo|cependant|pourtant)\b",
}

def marcar_patrones(df):
    t = df['text'].str.lower()
    return pd.DataFrame({k: t.str.contains(p, regex=True) for k, p in PATRONES.items()})

pat_train, pat_eval = marcar_patrones(train), marcar_patrones(eval_df)
tabla = pd.DataFrame({'train %': pat_train.mean() * 100, 'eval %': pat_eval.mean() * 100}).round(1)
display(tabla)
display((pd.concat([pat_train, train['label']], axis=1).groupby('label').mean() * 100).round(1).T
        .rename_axis('% de reseñas con el patrón, por clase'))

**Conclusiones de la exploración** *(revisar con los números obtenidos)*

1. **Clases casi balanceadas**, así que la exactitud es una métrica adecuada y no hace falta remuestrear.
2. **Cambio de distribución de idiomas:** `eval` tiene proporcionalmente **menos español y más inglés, portugués y francés** que `train`. Un modelo ajustado al español perdería exactitud en `eval`. Por eso usamos un modelo **multilingüe preentrenado**, que transfiere lo aprendido entre idiomas.
3. **El code-switching es muy escaso en train** (decenas de reseñas), pero aparece en `eval`. Crear mezclas sintéticas es la forma de que el modelo vea más ejemplos de ese tipo.
4. Las reseñas de `eval` son algo **más largas**, y los patrones difíciles (comparaciones, consecuencias, negaciones) son algo más frecuentes. Un modelo que procesa la secuencia completa y entiende el contexto (*transformer*) se ajusta mejor a esto que una bolsa de palabras.

### **3. Protocolo de validación**

- Partición **hold-out 90 % / 10 %**, estratificada por **idioma × clase**, así la validación tiene todos los idiomas en la misma proporción que train. Se usa *hold-out* y no validación cruzada porque un *fine-tuning* de XLM-R tarda decenas de minutos. Con ~2,000 reseñas de validación el error estándar de la exactitud es de ≈ 0.6 puntos: diferencias menores a eso entre dos experimentos pueden ser ruido (por eso se usa también la prueba de McNemar).
- **La aumentación se genera solo a partir de la partición de entrenamiento.** Si una copia modificada de una reseña de validación entrara al entrenamiento, el modelo vería su contenido y la exactitud de validación quedaría inflada (fuga de información).
- Además de la exactitud global se reporta una **exactitud ponderada a la distribución de idiomas de eval**: Σ p_eval(idioma) · acc_val(idioma). Es una estimación más realista del puntaje de Kaggle, porque eval tiene más reseñas no españolas.

In [ ]:
estrato = train['idioma'] + '_' + train['label']
estrato = estrato.where(estrato.map(estrato.value_counts()) >= 2, 'raro')   # combinaciones con 1 sola reseña
tr, va = train_test_split(train, test_size=0.10, stratify=estrato, random_state=SEMILLA)
tr, va = tr.reset_index(drop=True), va.reset_index(drop=True)

P_EVAL = eval_df['idioma'].value_counts(normalize=True)
print(f'entrenamiento: {len(tr):,} | validación: {len(va):,}')
display(pd.crosstab(va['idioma'], va['label'], margins=True).reindex(IDIOMAS + ['All']))

### **4. Aumentación de datos**

#### 4.1 Qué estrategias se consideraron

El anuncio del curso del 12/11/2026 aclara que **no se permiten arquitecturas *decoder-only* ni modelos generativos de texto**. Por eso se descartó la traducción automática con modelos neuronales (MarianMT, NLLB, etc., que son generativos *encoder-decoder*) y la generación de reseñas con LLMs. Las dos estrategias elegidas **no generan texto con ningún modelo**: recombinan texto real del conjunto de entrenamiento y usan diccionarios bilingües publicados.

| Estrategia | ¿Se usa? | Razón |
|---|---|---|
| **A. Code-switching léxico con diccionario bilingüe** | **Sí** | En train casi no hay reseñas mezcladas, pero en eval sí. Se reemplaza una fracción de las palabras de una reseña en español por su equivalente en inglés (y al revés para reseñas en inglés) con los diccionarios MUSE. La estructura de la frase (orden, negación, conectores) no cambia, así que **la etiqueta se conserva**. |
| **B. Inserción de oraciones neutras de otro idioma** | **Sí** | Las oraciones de las reseñas neutrales son descriptivas (envío, tamaño, garantía) y casi nunca tienen palabras de opinión (lo verificamos abajo). Insertar 1-2 de ellas, **en otro idioma**, en posiciones intermedias de una reseña no cambia su veredicto, y produce reseñas multilingües a nivel de oración y más texto en en / pt / fr. Se inserta en el medio, nunca al inicio ni al final, porque ahí suele estar el veredicto. |
| Traducción automática con redes (MarianMT) | No | Es un modelo generativo: lo prohíbe el anuncio del curso. |
| Barajar el orden de las oraciones | No | **Cambia la etiqueta**: el sentimiento depende de qué opinión va al final o al inicio (lo vimos en la Parte 1). |
| Sinónimos / borrado / inserción aleatoria (EDA) | No | Puede borrar una negación o cambiar "excelente" por algo neutro, y así invertir el sentimiento sin cambiar la etiqueta (ruido). |

**Diccionarios MUSE** (Conneau et al., 2018, *Word Translation Without Parallel Data*): diccionarios bilingües de referencia publicados por Facebook Research con licencia **CC BY-NC 4.0** (uso académico no comercial permitido). Se usan `es-en` y `en-es` y se toma la primera traducción de cada palabra.

In [ ]:
import unicodedata, urllib.request

def sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def cargar_muse(par):
    """Descarga (una vez) y carga un diccionario MUSE como {palabra_origen: primera_traducción}."""
    ruta = DIR_AUG / f'muse_{par}.txt'
    if not ruta.exists():
        locales = list(Path('/kaggle/input').rglob(f'{par}.txt')) if EN_KAGGLE else []
        if locales:
            ruta.write_bytes(locales[0].read_bytes())
        else:
            urllib.request.urlretrieve(f'https://dl.fbaipublicfiles.com/arrival/dictionaries/{par}.txt', ruta)
    dic = {}
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        partes = linea.split()
        if len(partes) == 2 and partes[0] != partes[1]:
            dic.setdefault(partes[0], partes[1])
            dic.setdefault(sin_tildes(partes[0]), partes[1])   # muchas reseñas vienen sin tildes ("compre", "llego")
    return dic

MUSE = {'es': cargar_muse('es-en'), 'en': cargar_muse('en-es')}
print({k: f'{len(v):,} entradas' for k, v in MUSE.items()})

In [ ]:
def dividir_oraciones(texto):
    return [s for s in re.split(r'(?<=[.!?…])\s+', str(texto).strip()) if s]

TOKEN_SUST = re.compile(r"\w+|[^\w\s]+|\s+")
# Palabras que nunca se sustituyen: negaciones y conectores contrastivos, que definen el sentimiento
PROTEGIDAS = {'no', 'not', 'nunca', 'never', 'ni', 'nor', 'nada', 'nothing', 'pero', 'but', 'aunque', 'although',
              'sin', 'without', 'tampoco', 'neither', 'embargo', 'however', 'nadie', 'nobody'}

def code_switch_lexico(texto, dic, p, rng):
    """Reemplaza cada palabra traducible (>= 3 letras, no protegida) por su traducción con probabilidad p."""
    salida = []
    for tok in TOKEN_SUST.findall(texto):
        low = tok.lower()
        if len(low) >= 3 and low not in PROTEGIDAS and low in dic and rng.random() < p:
            trad = dic[low]
            salida.append(trad.capitalize() if tok[0].isupper() else trad)
        else:
            salida.append(tok)
    return ''.join(salida)

def insertar_neutras(texto, idioma, banco, rng, max_ins=2):
    """Inserta 1-2 oraciones neutras de OTRO idioma en posiciones intermedias (ni al inicio ni al final)."""
    oraciones = dividir_oraciones(texto)
    if len(oraciones) < 2:
        return None
    otros = [k for k in banco if k != idioma]
    for _ in range(rng.integers(1, max_ins + 1)):
        donante = banco[otros[rng.integers(len(otros))]]
        pos = rng.integers(1, len(oraciones))            # entre la primera y la última oración
        oraciones.insert(pos, donante[rng.integers(len(donante))])
    return ' '.join(oraciones)

#### 4.2 Verificación: ¿las oraciones de las reseñas neutrales realmente no tienen opinión?

La estrategia B solo es válida si las oraciones que se insertan no tienen sentimiento. Se mide qué fracción de ellas contiene alguna palabra de opinión frecuente (lista multilingüe). El banco de oraciones se construye **solo con reseñas neutrales de la partición de entrenamiento**, y se descartan las oraciones que contengan alguna palabra de opinión.

In [ ]:
OPINION_RE = re.compile(r'(excelente|buen[oa]|mal[oa]\b|deficiente|decepcion|encant|recomiend|pésim|pesim|horrible|genial|'
                        r'great|good|bad|terrible|love|hate|awful|amazing|disappoint|'
                        r'ótim|otim|bom\b|ruim|péssim|decepcion|'
                        r'parfait|nul\b|déçu|decu|génial|super\b)', re.I)

banco, fraccion_opinion = {}, {}
for idi in ['es', 'en', 'pt', 'fr']:
    ors = [s for t in tr.loc[(tr['label'] == 'neutral') & (tr['idioma'] == idi), 'text'] for s in dividir_oraciones(t)]
    con_op = [bool(OPINION_RE.search(s)) for s in ors]
    fraccion_opinion[idi] = np.mean(con_op)
    banco[idi] = [s for s, c in zip(ors, con_op) if not c and len(s.split()) >= 4]
display(pd.DataFrame({'oraciones en el banco': {k: len(v) for k, v in banco.items()},
                      '% con palabra de opinión (descartadas)': {k: round(100 * v, 2) for k, v in fraccion_opinion.items()}}))
for idi in banco:
    print(idi, '->', banco[idi][0])

#### 4.3 Construcción de los conjuntos aumentados

- **A (code-switching léxico):** a cada reseña de entrenamiento en español o inglés se le genera, con probabilidad `P_A`, una copia con el `P_SUST` = 30 % de sus palabras traducidas al otro idioma. La copia queda marcada como `es-en`.
- **B (inserción de neutras):** a una fracción `P_B` de **todas** las reseñas de entrenamiento (cualquier idioma) se le genera una copia con 1-2 oraciones neutras de otro idioma intercaladas.
- Ambas solo usan la partición de entrenamiento (`tr`), nunca la de validación ni `eval`.

In [ ]:
P_A, P_SUST, P_B = 0.3, 0.3, 0.4
rng = np.random.default_rng(SEMILLA)

filas_a, filas_b = [], []
for r in tr.itertuples():
    if r.idioma in ('es', 'en') and rng.random() < P_A:
        nuevo = code_switch_lexico(r.text, MUSE[r.idioma], P_SUST, rng)
        if nuevo != r.text:
            filas_a.append({'id': f'{r.id}_cs', 'text': nuevo, 'label': r.label, 'idioma': 'es-en', 'origen': 'A_code_switching'})
    if r.idioma in banco and rng.random() < P_B:
        nuevo = insertar_neutras(r.text, r.idioma, banco, rng)
        if nuevo:
            filas_b.append({'id': f'{r.id}_ins', 'text': nuevo, 'label': r.label, 'idioma': r.idioma, 'origen': 'B_insercion'})

aug_a, aug_b = pd.DataFrame(filas_a), pd.DataFrame(filas_b)
for a in (aug_a, aug_b):
    a['y'] = a['label'].map(C2I)
print(f'A, code-switching léxico: {len(aug_a):,} reseñas nuevas')
print(f'B, inserción de neutras:  {len(aug_b):,} reseñas nuevas')

ej = tr[tr['idioma'] == 'es'].iloc[3]
print('\nORIGINAL       :', ej['text'])
print('A (es->en 30 %):', code_switch_lexico(ej['text'], MUSE['es'], P_SUST, np.random.default_rng(1)))
print('B (inserción)  :', insertar_neutras(ej['text'], 'es', banco, np.random.default_rng(1)))
ej = tr[tr['idioma'] == 'en'].iloc[3]
print('\nORIGINAL       :', ej['text'])
print('A (en->es 30 %):', code_switch_lexico(ej['text'], MUSE['en'], P_SUST, np.random.default_rng(1)))

In [ ]:
tr['origen'] = 'original'
CONJUNTOS = {
    'sin_aug': tr,
    'A':       pd.concat([tr, aug_a], ignore_index=True),
    'A_B':     pd.concat([tr, aug_a, aug_b], ignore_index=True),
}
resumen = pd.DataFrame({k: v['idioma'].value_counts(normalize=True) * 100 for k, v in CONJUNTOS.items()}).reindex(IDIOMAS).fillna(0)
resumen['eval'] = P_EVAL.reindex(IDIOMAS).fillna(0) * 100
resumen.loc['n filas'] = [len(v) for v in CONJUNTOS.values()] + [len(eval_df)]
display(resumen.round(1).rename_axis('% por idioma'))

La aumentación A aumenta mucho el número de reseñas `es-en`, que en train eran apenas unas decenas. La B agrega mezclas de idiomas a nivel de oración en todos los idiomas, incluidos pt y fr.

### **5. Experimentos**

Se cambia **una cosa a la vez** para poder atribuir cada mejora a una decisión. Cada experimento genera un archivo de envío en `envios/`; así quedan los ≥ 5 envíos diferentes y cada uno corresponde a un paso documentado.

| Exp. | Modelo | Datos de entrenamiento | Qué se quiere medir |
|---|---|---|---|
| E1 | BiLSTM desde cero | sin aumentación | punto de partida secuencial sin preentrenamiento |
| E2 | BiLSTM desde cero | A + B | efecto de la aumentación sin preentrenamiento |
| E3 | XLM-R **base** (fine-tuning) | sin aumentación | efecto del preentrenamiento multilingüe |
| E4 | XLM-R base | + A (code-switching léxico) | efecto de la aumentación A |
| E5 | XLM-R base | + A + B (inserción de neutras) | efecto de la aumentación B |
| E6 | XLM-R **large** | mejor conjunto de E3-E5 | efecto del tamaño del *encoder* |
| E7 | XLM-R large, otra semilla | mejor conjunto de E3-E5 | variabilidad entre semillas |
| E8 | **Ensamble** E6 + E7 | (promedio de probabilidades) | efecto de combinar modelos |
| E9 | Ensamble E6 + E7 + mejor base | (promedio de probabilidades) | si un modelo distinto aporta diversidad |

La selección del conjunto de datos se hace con el modelo *base* (más barato) y solo la mejor configuración se entrena con el modelo *large*.

#### 5.1 Utilidades de evaluación

In [ ]:
RESULTADOS_CSV = DIR_RES / 'resultados.csv'
resultados = pd.read_csv(RESULTADOS_CSV).to_dict('records') if (RESULTADOS_CSV.exists() and REUTILIZAR_CACHE) else []

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def acc_por_idioma(df, pred):
    ok = (df['y'].values == pred)
    return pd.Series(ok, index=df.index).groupby(df['idioma']).mean()

def registrar(nombre, descripcion, p_val, p_eval, segundos):
    """Guarda probabilidades, calcula métricas y escribe el archivo de envío del experimento."""
    np.save(DIR_RES / f'{nombre}_val.npy', p_val)
    np.save(DIR_RES / f'{nombre}_eval.npy', p_eval)
    pred = p_val.argmax(1)
    por_idi = acc_por_idioma(va, pred)
    fila = {
        'experimento': nombre, 'descripcion': descripcion,
        'acc_val': accuracy_score(va['y'], pred),
        'acc_ponderada_eval': float((por_idi * P_EVAL).dropna().sum() / P_EVAL[por_idi.index].sum()),
        **{f'acc_{k}': v for k, v in por_idi.items()},
        'minutos': segundos / 60,
    }
    global resultados
    resultados = [r for r in resultados if r['experimento'] != nombre] + [fila]
    pd.DataFrame(resultados).to_csv(RESULTADOS_CSV, index=False)
    envio = pd.DataFrame({'id': eval_df['id'], 'answer': [CLASES[i] for i in p_eval.argmax(1)]})
    assert len(envio) == len(sample) and envio['id'].equals(sample['id'])
    envio.to_csv(DIR_ENV / f'{nombre}.csv', index=False)
    print(f"{nombre}: acc_val={fila['acc_val']:.4f} | ponderada_eval={fila['acc_ponderada_eval']:.4f} | "
          + ' '.join(f'{k}={v:.3f}' for k, v in por_idi.items()))
    return fila

def ya_calculado(nombre):
    return REUTILIZAR_CACHE and (DIR_RES / f'{nombre}_val.npy').exists() and any(r['experimento'] == nombre for r in resultados)

def cargar_probs(nombre):
    return np.load(DIR_RES / f'{nombre}_val.npy'), np.load(DIR_RES / f'{nombre}_eval.npy')

def acc_de(nombre):
    return next((r['acc_val'] for r in resultados if r['experimento'] == nombre), float('nan'))

#### 5.2 Línea de comparación: BiLSTM desde cero (E1, E2)

Arquitectura: `TextVectorization` (vocabulario aprendido de train) → `Embedding` 128 → **BiLSTM** 96 por dirección → *global max pooling* → `Dense` 64 → 3 *logits*.

A diferencia de la bolsa de palabras de la Parte 1, la BiLSTM **procesa la secuencia en orden**, así que puede capturar dónde está cada opinión y el alcance de una negación. Pero no tiene conocimiento previo del idioma: todo lo aprende de ~18 mil reseñas, y en idiomas con pocos ejemplos (fr, pt) eso es poco. Sirve para medir cuánto aporta el preentrenamiento.

In [ ]:
VOCAB, LARGO_PAL = 40000, 160

def construir_bilstm(textos_entrenamiento):
    vec = keras.layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LARGO_PAL)
    vec.adapt(tf.data.Dataset.from_tensor_slices(textos_entrenamiento).batch(512))
    ent = keras.Input(shape=(1,), dtype='string')
    x = vec(ent)
    x = keras.layers.Embedding(VOCAB, 128)(x)
    x = keras.layers.SpatialDropout1D(0.2)(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(96, return_sequences=True))(x)
    x = keras.layers.GlobalMaxPooling1D()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    sal = keras.layers.Dense(3, dtype='float32')(x)          # logits en float32 (estable con precisión mixta)
    modelo = keras.Model(ent, sal)
    modelo.compile(optimizer=keras.optimizers.Adam(2e-3),
                   loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    return modelo

def experimento_bilstm(nombre, descripcion, df_entrenamiento, epocas=10):
    if ya_calculado(nombre):
        print(nombre, 'ya calculado: se reutiliza'); return
    keras.utils.set_random_seed(SEMILLA)
    t0 = time.time()
    X = df_entrenamiento['text'].values.astype(str)
    modelo = construir_bilstm(X)
    pesos = str(DIR_MOD / f'{nombre}.weights.h5')
    hist = modelo.fit(X[:, None], df_entrenamiento['y'].values,
                      validation_data=(va['text'].values.astype(str)[:, None], va['y'].values),
                      epochs=epocas, batch_size=128, verbose=2,
                      callbacks=[keras.callbacks.ModelCheckpoint(pesos, monitor='val_accuracy', mode='max',
                                                                 save_best_only=True, save_weights_only=True),
                                 keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=3)])
    modelo.load_weights(pesos)
    p_val = softmax(np.asarray(modelo.predict(va['text'].values.astype(str)[:, None], batch_size=512, verbose=0), dtype='float32'))
    p_eval = softmax(np.asarray(modelo.predict(eval_df['text'].values.astype(str)[:, None], batch_size=512, verbose=0), dtype='float32'))
    registrar(nombre, descripcion, p_val, p_eval, time.time() - t0)
    pd.DataFrame(hist.history).to_csv(DIR_RES / f'{nombre}_historia.csv', index=False)
    del modelo; keras.backend.clear_session(); gc.collect()

experimento_bilstm('E1_bilstm_sin_aug', 'BiLSTM desde cero, sin aumentación', CONJUNTOS['sin_aug'])
experimento_bilstm('E2_bilstm_aug', 'BiLSTM desde cero, + A + B', CONJUNTOS['A_B'])

#### 5.3 Transferencia de aprendizaje: XLM-RoBERTa (E3-E7)

**Modelo:** XLM-RoBERTa (Conneau et al., 2020), un *encoder-only* preentrenado con *masked language modeling* sobre 2.5 TB de texto CommonCrawl en 100 idiomas. Se carga desde **KerasHub** (licencia MIT) en dos tamaños:

| *Preset* | Capas | Dimensión | Parámetros |
|---|---|---|---|
| `xlm_roberta_base_multi` | 12 | 768 | ≈ 277 M |
| `xlm_roberta_large_multi` | 24 | 1024 | ≈ 558 M |

`TextClassifier` agrega sobre el *encoder* una cabeza de clasificación (capa densa + *dropout* + 3 *logits*) que usa la representación del token `<s>`, y se ajusta (*fine-tuning*) **todo** el modelo. Esto corresponde a lo que permite el curso: embeddings contextuales de un *encoder-only* más una cabeza de clasificación.

**Por qué este modelo:**
- Un mismo *encoder* para los 4 idiomas y para el texto mezclado, con un vocabulario SentencePiece compartido. Lo aprendido en español se transfiere a portugués o francés.
- La autoatención mira la reseña completa a la vez, así que puede relacionar una concesión al inicio con el veredicto, o "no diría que" con el adjetivo que viene varias palabras después. El error de los modelos clásicos está casi todo en negativo ↔ positivo, que depende justo de eso.
- El tamaño *large* tiene más capacidad para los casos de ironía y negación larga. Se entrena solo con la mejor configuración de datos porque cuesta ~3 veces más.

**Hiperparámetros** (valores estándar para *fine-tuning* de modelos tipo BERT):
- AdamW con *warmup* lineal del 10 % y decaimiento coseno, *weight decay* 0.01, *clipping* del gradiente a norma 1.
- *base*: lr 2e-5, lote 32, 4 épocas. *large*: lr 1e-5, lote 16, 3 épocas (un lr más bajo porque los modelos grandes son más inestables).
- Longitud máxima `MAX_LEN` tokens, calculada abajo para cubrir ~99.5 % de las reseñas **sin truncarlas**: el veredicto puede estar al inicio o al final, así que no conviene cortar ninguno de los dos extremos.
- Se guarda el **mejor checkpoint según la exactitud de validación**.
- **Control de colapso:** a veces el *fine-tuning* de un modelo grande no arranca y se queda prediciendo una sola clase. Si después de la primera época la exactitud de validación es menor a 0.5, se detiene y se reintenta con otra semilla y un lr menor.

In [ ]:
PRESET_BASE, PRESET_LARGE = 'xlm_roberta_base_multi', 'xlm_roberta_large_multi'
tokenizador = keras_hub.models.TextClassifierPreprocessor.from_preset(PRESET_BASE).tokenizer
ids = tokenizador(list(pd.concat([train['text'], eval_df['text']]).sample(4000, random_state=SEMILLA)))
largos = ids.row_lengths().numpy() if hasattr(ids, 'row_lengths') else np.array([len(x) for x in ids])
MAX_LEN = int(min(256, math.ceil((np.percentile(largos, 99.5) + 2) / 32) * 32))
print(f'Tokens por reseña: media={largos.mean():.0f}, p95={np.percentile(largos, 95):.0f}, máx={largos.max()} -> MAX_LEN={MAX_LEN}')

In [ ]:
def construir_transformer(preset, lr, pasos_totales, max_len=None):
    max_len = max_len or MAX_LEN
    pre = keras_hub.models.TextClassifierPreprocessor.from_preset(preset, sequence_length=max_len)
    clf = keras_hub.models.TextClassifier.from_preset(preset, num_classes=3, preprocessor=pre)
    calentamiento = max(1, int(0.1 * pasos_totales))
    horario = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0, warmup_target=lr, warmup_steps=calentamiento,
        decay_steps=max(1, pasos_totales - calentamiento))
    opt = keras.optimizers.AdamW(learning_rate=horario, weight_decay=0.01, global_clipnorm=1.0)
    clf.compile(optimizer=opt, loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
    return clf

class DetenerSiColapsa(keras.callbacks.Callback):
    """Detiene el entrenamiento si tras la primera época el modelo no aprendió (predice casi al azar)."""
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 and (logs or {}).get('val_accuracy', 1.0) < 0.5:
            print('  -> colapso detectado: se detiene este intento')
            self.model.stop_training = True

def predecir_probs(modelo, textos, lote=64):
    return softmax(np.asarray(modelo.predict(np.asarray(textos, dtype=str), batch_size=lote, verbose=0), dtype='float32'))

def experimento_transformer(nombre, descripcion, df_entrenamiento, preset=PRESET_BASE, lr=2e-5,
                            epocas=4, lote=32, max_len=None, semilla=SEMILLA):
    if ya_calculado(nombre):
        print(nombre, 'ya calculado: se reutiliza'); return
    t0 = time.time()
    pesos = str(DIR_MOD / f'{nombre}.weights.h5')
    X, y = df_entrenamiento['text'].values.astype(str), df_entrenamiento['y'].values
    Xv, yv = va['text'].values.astype(str), va['y'].values
    for intento in range(3):
        keras.backend.clear_session(); gc.collect()
        keras.utils.set_random_seed(semilla + intento)
        lr_i = lr * (0.7 ** intento)
        clf = construir_transformer(preset, lr_i, math.ceil(len(X) / lote) * epocas, max_len)
        hist = clf.fit(x=X, y=y, validation_data=(Xv, yv), batch_size=lote, epochs=epocas, verbose=1,
                       callbacks=[keras.callbacks.ModelCheckpoint(pesos, monitor='val_accuracy', mode='max',
                                                                  save_best_only=True, save_weights_only=True),
                                  DetenerSiColapsa()])
        if max(hist.history['val_accuracy']) >= 0.5 or intento == 2:
            break
        del clf
    clf.load_weights(pesos)
    p_val, p_eval = predecir_probs(clf, Xv), predecir_probs(clf, eval_df['text'].values)
    registrar(nombre, descripcion, p_val, p_eval, time.time() - t0)
    pd.DataFrame(hist.history).to_csv(DIR_RES / f'{nombre}_historia.csv', index=False)
    json.dump({'preset': preset, 'lr': lr_i, 'epocas': epocas, 'lote': lote, 'max_len': max_len or MAX_LEN,
               'semilla': semilla + intento}, open(DIR_MOD / f'{nombre}.json', 'w'))
    del clf; keras.backend.clear_session(); gc.collect()

**E3-E5: XLM-R base, con y sin aumentación**

In [ ]:
experimento_transformer('E3_xlmr_sin_aug', 'XLM-R base, sin aumentación', CONJUNTOS['sin_aug'])

In [ ]:
experimento_transformer('E4_xlmr_A', 'XLM-R base, + A (code-switching léxico)', CONJUNTOS['A'])

In [ ]:
experimento_transformer('E5_xlmr_A_B', 'XLM-R base, + A + B (inserción de neutras)', CONJUNTOS['A_B'])

**E6-E7: XLM-R large con el mejor conjunto de datos**

El conjunto de datos se elige con la exactitud de validación de E3-E5. Se entrenan dos modelos *large* que solo se diferencian en la semilla (inicialización de la cabeza y orden de los lotes). Sirven para dos cosas: medir cuánto varía el resultado por azar y tener dos modelos para el ensamble.

In [ ]:
EXP_DE = {'sin_aug': 'E3_xlmr_sin_aug', 'A': 'E4_xlmr_A', 'A_B': 'E5_xlmr_A_B'}
MEJOR_DATOS = max(EXP_DE, key=lambda k: acc_de(EXP_DE[k]) if not np.isnan(acc_de(EXP_DE[k])) else -1)
MEJOR_BASE = EXP_DE[MEJOR_DATOS]
print({k: round(acc_de(v), 4) for k, v in EXP_DE.items()}, '-> mejor conjunto de datos:', MEJOR_DATOS)

In [ ]:
experimento_transformer('E6_xlmr_large', f'XLM-R large, datos {MEJOR_DATOS}', CONJUNTOS[MEJOR_DATOS],
                        preset=PRESET_LARGE, lr=1e-5, lote=16, epocas=3, semilla=SEMILLA)

In [ ]:
experimento_transformer('E7_xlmr_large_s2', f'XLM-R large, datos {MEJOR_DATOS}, semilla 2', CONJUNTOS[MEJOR_DATOS],
                        preset=PRESET_LARGE, lr=1e-5, lote=16, epocas=3, semilla=2026)

#### 5.4 Ensamble (E8, E9)

Dos modelos entrenados con semillas distintas se equivocan en reseñas parcialmente distintas. Al **promediar sus probabilidades**, los errores que no comparten tienden a cancelarse: es la misma idea del *bagging* y del *stacking* de la Parte 1, aplicada a redes. No se entrena nada nuevo: solo se promedian las probabilidades ya guardadas de cada modelo.

- **E8:** los dos XLM-R large.
- **E9:** los dos large más el mejor XLM-R base, para ver si un modelo de otro tamaño aporta diversidad o solo baja el promedio.

In [ ]:
MIEMBROS_ENSAMBLE = {}

def ensamble(nombre, descripcion, miembros):
    miembros = [m for m in miembros if (DIR_RES / f'{m}_val.npy').exists()]
    if len(miembros) < 2:
        print(nombre, ': faltan miembros, se omite'); return
    p_val = np.mean([cargar_probs(m)[0] for m in miembros], axis=0)
    p_eval = np.mean([cargar_probs(m)[1] for m in miembros], axis=0)
    MIEMBROS_ENSAMBLE[nombre] = miembros
    registrar(nombre, descripcion, p_val, p_eval, 0)

ensamble('E8_ensamble_large', 'Promedio de E6 + E7', ['E6_xlmr_large', 'E7_xlmr_large_s2'])
ensamble('E9_ensamble_large_base', f'Promedio de E6 + E7 + {MEJOR_BASE}', ['E6_xlmr_large', 'E7_xlmr_large_s2', MEJOR_BASE])

### **6. Comparación de experimentos y efecto de la aumentación**

In [ ]:
tabla_res = pd.DataFrame(resultados).sort_values('experimento').set_index('experimento')
cols_idi = [f'acc_{i}' for i in IDIOMAS if f'acc_{i}' in tabla_res]
display(tabla_res[['descripcion', 'acc_val', 'acc_ponderada_eval', *cols_idi, 'minutos']]
        .style.format({c: '{:.4f}' for c in ['acc_val', 'acc_ponderada_eval', *cols_idi]} | {'minutos': '{:.1f}'})
        .background_gradient(subset=['acc_val'], cmap='Greens'))

ax = tabla_res[cols_idi].rename(columns=lambda c: c[4:]).plot.bar(figsize=(15, 4), rot=30)
ax.set_ylim(max(0, tabla_res[cols_idi].min().min() - 0.05), 1)
ax.set_title('Exactitud de validación por idioma y experimento'); ax.legend(ncol=5)
plt.tight_layout(); plt.show()

**¿Las diferencias son reales o ruido?** Se usa la **prueba de McNemar** sobre las mismas reseñas de validación. Solo cuentan las reseñas en las que los dos modelos discrepan; si el cambio no tuviera efecto, cada modelo acertaría la mitad de esas discrepancias. La comparación E6 vs E7 (misma configuración, distinta semilla) muestra cuánta diferencia produce solo el azar.

In [ ]:
def mcnemar(a, b):
    pa, pb = cargar_probs(a)[0].argmax(1), cargar_probs(b)[0].argmax(1)
    ok_a, ok_b = pa == va['y'].values, pb == va['y'].values
    solo_a, solo_b = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    p = binomtest(solo_b, solo_a + solo_b, 0.5).pvalue if solo_a + solo_b else 1.0
    print(f'{b} vs {a}: Δacc = {ok_b.mean() - ok_a.mean():+.4f} | solo acierta {a}: {solo_a}, '
          f'solo acierta {b}: {solo_b} | p-valor McNemar = {p:.4f}')

hechos = set(tabla_res.index)
print('--- Con vs sin aumentación ---')
for a, b in [('E1_bilstm_sin_aug', 'E2_bilstm_aug'), ('E3_xlmr_sin_aug', 'E4_xlmr_A'),
             ('E4_xlmr_A', 'E5_xlmr_A_B'), ('E3_xlmr_sin_aug', 'E5_xlmr_A_B')]:
    if {a, b} <= hechos:
        mcnemar(a, b)
print('--- Preentrenamiento, tamaño, semilla y ensamble ---')
for a, b in [('E1_bilstm_sin_aug', 'E3_xlmr_sin_aug'), (MEJOR_BASE, 'E6_xlmr_large'),
             ('E6_xlmr_large', 'E7_xlmr_large_s2'), ('E6_xlmr_large', 'E8_ensamble_large'),
             ('E8_ensamble_large', 'E9_ensamble_large_base')]:
    if {a, b} <= hechos:
        mcnemar(a, b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n in tabla_res.index:
    h = DIR_RES / f'{n}_historia.csv'
    if h.exists():
        hh = pd.read_csv(h)
        axes[0].plot(range(1, len(hh) + 1), hh['val_accuracy'], marker='o', label=n)
        axes[1].plot(range(1, len(hh) + 1), hh['loss'], marker='o', label=n)
axes[0].set_title('Exactitud de validación por época'); axes[1].set_title('Pérdida de entrenamiento por época')
for a in axes: a.set_xlabel('época'); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Análisis** *(completar con los resultados obtenidos)*

- **Preentrenamiento (E1 → E3):** cuánto mejora XLM-R frente a la BiLSTM desde cero, sobre todo en fr / pt / es-en.
- **Code-switching léxico (E3 → E4):** efecto global y por idioma, sobre todo en `es-en` (pocas reseñas en validación, así que mirar también el p-valor) y en la exactitud ponderada a eval.
- **Inserción de neutras (E4 → E5):** si hacer al modelo robusto a oraciones de relleno en otro idioma mejora pt / fr / es-en.
- **Aumentación en la BiLSTM (E1 → E2):** sin preentrenamiento la BiLSTM depende más de la cantidad de datos. Comparar si la ganancia es mayor o menor que en XLM-R.
- **Tamaño del modelo (base → large, E6):** cuánto sube y en qué idiomas o tipos de reseña.
- **Semilla (E6 vs E7):** esa diferencia es el "ruido" contra el que hay que comparar todas las demás mejoras.
- **Ensamble (E8, E9):** cuánto gana sobre el mejor modelo individual y si agregar el modelo base ayuda.
- Comparar el ranking de validación con el *score* público de Kaggle de cada envío.

### **7. Análisis de errores por idioma**

Sobre el mejor experimento (mayor exactitud de validación), se analiza la exactitud y la matriz de confusión por idioma, la exactitud por tipo de reseña difícil y ejemplos concretos de errores.

In [ ]:
MEJOR = tabla_res['acc_val'].idxmax()
p_val_mejor = cargar_probs(MEJOR)[0]
va_err = va.assign(pred=p_val_mejor.argmax(1), conf=p_val_mejor.max(1))
va_err['acierto'] = va_err['pred'] == va_err['y']
print('Mejor experimento:', MEJOR)
print(classification_report(va_err['y'], va_err['pred'], target_names=CLASES, digits=4))

por_idioma = va_err.groupby('idioma').agg(n=('acierto', 'size'), exactitud=('acierto', 'mean')).reindex(IDIOMAS)
por_idioma['errores'] = (por_idioma['n'] * (1 - por_idioma['exactitud'])).round().astype('Int64')
display(por_idioma.style.format({'exactitud': '{:.4f}'}))

fig, axes = plt.subplots(1, len(IDIOMAS), figsize=(4 * len(IDIOMAS), 3.6))
for ax, idi in zip(axes, IDIOMAS):
    sub = va_err[va_err['idioma'] == idi]
    if len(sub):
        ConfusionMatrixDisplay(confusion_matrix(sub['y'], sub['pred'], labels=[0, 1, 2]),
                               display_labels=['neg', 'neu', 'pos']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{idi} (n={len(sub)}, acc={sub["acierto"].mean():.3f})')
plt.tight_layout(); plt.show()

In [ ]:
pat_va = marcar_patrones(va_err)
tipo = pd.DataFrame({
    'n': pat_va.sum(),
    'exactitud': [va_err.loc[pat_va[c], 'acierto'].mean() for c in pat_va],
}).sort_values('exactitud')
tipo.loc['(sin ningún patrón)'] = [int((~pat_va.any(axis=1)).sum()), va_err.loc[~pat_va.any(axis=1), 'acierto'].mean()]
display(tipo.style.format({'exactitud': '{:.4f}'}).set_caption('Exactitud por tipo de reseña'))

errores = va_err[~va_err['acierto']].copy()
errores['real'] = errores['y'].map(dict(enumerate(CLASES)))
errores['predicho'] = errores['pred'].map(dict(enumerate(CLASES)))
print('Tipos de confusión más frecuentes:')
display(errores.groupby(['real', 'predicho']).size().sort_values(ascending=False).to_frame('n'))

# ¿El modelo sabe cuándo duda? Exactitud según la confianza de la predicción
va_err['tramo_conf'] = pd.cut(va_err['conf'], [0, 0.6, 0.8, 0.95, 1.0])
display(va_err.groupby('tramo_conf', observed=True).agg(n=('acierto', 'size'), exactitud=('acierto', 'mean')).round(4))

In [ ]:
for idi in IDIOMAS:
    sub = errores[errores['idioma'] == idi].sort_values('conf', ascending=False).head(3)
    if len(sub):
        print(f'\n=========== {idi}: errores con mayor confianza ===========')
        for r in sub.itertuples():
            print(f'[real={r.real} | pred={r.predicho} | conf={r.conf:.2f}]\n  {r.text}\n')

**Análisis de errores** *(completar después de ver los ejemplos)*

- ¿En qué idioma se equivoca más el modelo y por qué? (pocos ejemplos de entrenamiento, mezcla de idiomas).
- ¿Qué confusión domina? En la Parte 1 casi todos los errores eran negativo ↔ positivo en reseñas mixtas y la clase neutral estaba resuelta. Ver si se mantiene.
- ¿Qué tipo de reseña difícil (ironía, comparación, negación larga, consecuencia) concentra los errores?
- Errores que parecen **etiquetas ambiguas o ruidosas**: en la exploración vimos reseñas "indecisas" ("no me decido todavía") con etiquetas que no siguen una regla clara. Ese error no se puede reducir con un mejor modelo.
- Qué se haría con más tiempo (p. ej. más semillas en el ensamble, reentrenar con train + validación).

### **8. Modelo final, guardado y envío**

El modelo final es el experimento con mayor exactitud de validación; debe coincidir con el envío de mejor *score* público en Kaggle (si no coinciden, cambiar `FINAL` a mano y justificarlo).

- Si el mejor es un **ensamble**, el modelo final son sus miembros: se guarda un archivo `.keras` por miembro y la predicción es el promedio de sus probabilidades.
- Cada archivo `.keras` contiene el modelo **completo** (arquitectura + pesos + tokenizador), así que recibe texto crudo.
- Se verifica que cada archivo, cargado de nuevo, reproduce las predicciones del experimento.
- Al final se borran los pesos intermedios (`.weights.h5`) para no llenar el disco de Kaggle.

In [ ]:
FINAL = MEJOR          # cambiar a mano si el mejor score público de Kaggle es otro experimento
MIEMBROS = MIEMBROS_ENSAMBLE.get(FINAL, [FINAL])
print('Modelo final:', FINAL, '| miembros:', MIEMBROS)

probs_eval = []
for i, m in enumerate(MIEMBROS, 1):
    keras.backend.clear_session(); gc.collect()
    ruta = DIR_MOD / (f'modelo_final_{i}_{m}.keras' if len(MIEMBROS) > 1 else 'modelo_final.keras')
    if not ruta.exists():
        if m.startswith('E1') or m.startswith('E2'):
            conj = CONJUNTOS['sin_aug'] if m.startswith('E1') else CONJUNTOS['A_B']
            modelo = construir_bilstm(conj['text'].values.astype(str))
        else:
            cfg = json.load(open(DIR_MOD / f'{m}.json'))
            modelo = construir_transformer(cfg['preset'], cfg['lr'], 1000, cfg['max_len'])
        modelo.load_weights(str(DIR_MOD / f'{m}.weights.h5'))
        modelo.save(ruta)
        del modelo; keras.backend.clear_session(); gc.collect()
    # Verificación de reproducibilidad: cargar desde el archivo y predecir eval
    recargado = keras.models.load_model(ruta, compile=False)
    X_eval = np.asarray(eval_df['text'].values, dtype=str)
    if m.startswith('E1') or m.startswith('E2'):
        X_eval = X_eval[:, None]
    p = softmax(np.asarray(recargado.predict(X_eval, batch_size=64, verbose=0), dtype='float32'))
    iguales = (p.argmax(1) == cargar_probs(m)[1].argmax(1)).mean()
    print(f'{ruta.name}: {ruta.stat().st_size / 1e6:.0f} MB | predicciones iguales a las del experimento: {iguales:.2%}')
    probs_eval.append(p)
    del recargado

p_eval_final = np.mean(probs_eval, axis=0)
envio_final = pd.DataFrame({'id': eval_df['id'], 'answer': [CLASES[i] for i in p_eval_final.argmax(1)]})
envio_final.to_csv(DIR_ENV / 'envio_final.csv', index=False)
print('Coincidencia de envio_final con el envío del experimento:',
      f"{(envio_final['answer'] == pd.read_csv(DIR_ENV / f'{FINAL}.csv')['answer']).mean():.2%}")
display(envio_final['answer'].value_counts(normalize=True).round(3).to_frame('proporción predicha en eval'))

LIMPIAR_PESOS = True
if LIMPIAR_PESOS:
    for f in DIR_MOD.glob('*.weights.h5'):
        f.unlink()

**Cómo usar el modelo entregado** (no requiere este notebook):

```python
import glob, keras, keras_hub, numpy as np
from scipy.special import softmax

textos = np.array(['Lo devolví a la semana.', 'Works great, love it!'])
archivos = sorted(glob.glob('modelo_final*.keras'))          # 1 archivo, o varios si es un ensamble
probs = np.mean([softmax(keras.models.load_model(a, compile=False).predict(textos).astype('float32'), axis=1)
                 for a in archivos], axis=0)
print(np.array(['negativo', 'neutral', 'positivo'])[probs.argmax(1)])
```

### **9. Conclusiones** *(completar con los resultados)*

1. **Modelo final:** ... (exactitud validación ..., ponderada a eval ..., score público Kaggle ...).
2. **Lo que más aportó:** ... (preentrenamiento, tamaño del modelo, aumentación, ensamble: ordenar por ganancia medida).
3. **Efecto de la aumentación:** con vs sin, global y por idioma, con el p-valor de McNemar.
4. **Errores restantes:** ...
5. **Comparación con la Parte 1:** un TF-IDF + regresión logística sobre los datos nuevos llega a ≈ 0.72 en validación cruzada (la clase neutral queda resuelta y casi todo el error es negativo ↔ positivo). Comparar con la BiLSTM y XLM-R.

### Registro de envíos a Kaggle

| # | Archivo (`envios/`) | Experimento | Acc. validación | Score público Kaggle |
|---|---|---|---|---|
| 1 | E1_bilstm_sin_aug.csv | BiLSTM sin aug | | |
| 2 | E2_bilstm_aug.csv | BiLSTM + A + B | | |
| 3 | E3_xlmr_sin_aug.csv | XLM-R base sin aug | | |
| 4 | E4_xlmr_A.csv | XLM-R base + A | | |
| 5 | E5_xlmr_A_B.csv | XLM-R base + A + B | | |
| 6 | E6_xlmr_large.csv | XLM-R large | | |
| 7 | E7_xlmr_large_s2.csv | XLM-R large, semilla 2 | | |
| 8 | E8_ensamble_large.csv | Ensamble E6 + E7 | | |
| 9 | E9_ensamble_large_base.csv | Ensamble E6 + E7 + base | | |
| 10 | envio_final.csv | modelo final | | |